# NLI base results: Qwen3-1.7B (qwen3:1.7b via Ollama)

Loads [yilmazzey/sdp2-nli](https://huggingface.co/datasets/yilmazzey/sdp2-nli) (snli_tr_1_1, multinli_tr_1_1, trglue_mnli) and runs **test-only** evaluation with this model.

1.7B generative LLM (Qwen3 series). Multilingual support. Zero-shot prompted NLI via Ollama (no fine-tuning). Outputs parsed to 0=entailment, 1=neutral, 2=contradiction.

**Splits:** snli → test; multinli → validation_matched/mismatched; trglue → test_matched/test_mismatched. **Metrics:** Accuracy, macro F1, per-class F1, confusion matrix (CSV + plot).

In [1]:
# Install dependencies (run once)
!pip install -q -U datasets scikit-learn tqdm ollama

import json
import random
from collections import Counter
from pathlib import Path

import numpy as np
from datasets import load_dataset
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from tqdm import tqdm
import ollama

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_PLOT = True
except ImportError:
    HAS_PLOT = False

LABEL_NAMES = ["entailment", "neutral", "contradiction"]

print("Using Ollama for local LLM inference (qwen3:1.7b)")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using Ollama for local LLM inference (qwen3:1.7b)


In [2]:
REPO_ID = "yilmazzey/sdp2-nli"
CONFIGS = ["snli_tr_1_1", "multinli_tr_1_1", "trglue_mnli"]
MODEL_ID = "qwen3:1.7b"
NUM_LABELS = 3  # entailment, neutral, contradiction
RESULTS_DIR = "results"
# Ollama processes one at a time
BATCH_SIZE = 1
EVAL_SPLITS = {
    "snli_tr_1_1": ["test"],
    "multinli_tr_1_1": ["validation_matched", "validation_mismatched"],
    "trglue_mnli": ["test_matched", "test_mismatched"],
}

In [3]:
# Load all three dataset configs
datasets = {}
for cfg in CONFIGS:
    print(f"Loading {REPO_ID} :: {cfg} ...")
    datasets[cfg] = load_dataset(REPO_ID, cfg)
    print("  splits:", list(datasets[cfg].keys()))

Loading yilmazzey/sdp2-nli :: snli_tr_1_1 ...
  splits: ['train', 'validation', 'test']
Loading yilmazzey/sdp2-nli :: multinli_tr_1_1 ...
  splits: ['train', 'validation_matched', 'validation_mismatched']
Loading yilmazzey/sdp2-nli :: trglue_mnli ...
  splits: ['train', 'validation_matched', 'validation_mismatched', 'test_matched', 'test_mismatched']


In [4]:
print(f"Using Ollama model: {MODEL_ID}")
print("Make sure the model is pulled: ollama pull qwen3:1.7b")

# Test Ollama connection
try:
    response = ollama.chat(
        model=MODEL_ID,
        messages=[{"role": "user", "content": "test"}],
        options={"temperature": 0.0, "num_predict": 1, "thinking": False}
    )
    print("Ollama connection successful.")
except Exception as e:
    print(f"Error connecting to Ollama: {e}")
    raise

Using Ollama model: qwen3:1.7b
Make sure the model is pulled: ollama pull qwen3:1.7b
Ollama connection successful.


In [5]:
def nli_prompt(premise, hypothesis):
    return [
        {"role": "system", "content": "You are a helpful assistant for natural language inference. Classify the relationship between premise and hypothesis as entailment, neutral, or contradiction. Respond with exactly one word only: entailment, neutral, or contradiction. No explanation, no other text."},
        {"role": "user", "content": f"Premise: {premise}\nHypothesis: {hypothesis}\nLabel:"}
    ]

def parse_generated_label(gen_text):
    # Clean and parse the generated text
    continuation = gen_text.strip().lower()
    if not continuation:
        return 1  # neutral fallback

    # Take first word, remove punctuation
    first_word = continuation.split()[0].rstrip('.,!?;:')

    label_map = {"entailment": 0, "neutral": 1, "contradiction": 2}
    return label_map.get(first_word, 1)  # Default to neutral if unknown

def run_prompted_inference(ds):
    premises = list(ds["premise"])
    hypotheses = list(ds["hypothesis"])
    labels = list(ds["label"])
    n = len(ds)
    y_pred = []
    all_generations = []  # Collect all for full debug

    for i in tqdm(range(n), desc="Inference"):
        messages = nli_prompt(premises[i], hypotheses[i])
        
        try:
            response = ollama.chat(
                model=MODEL_ID,
                messages=messages,
                options={
                    "temperature": 0.0,
                    "num_predict": 5,  # Very strict to force single word
                    "thinking": False  # CRITICAL: Disable thinking for Qwen3
                }
            )
            gen_text = response['message']['content']
            all_generations.append(gen_text)
            parsed = parse_generated_label(gen_text)
            y_pred.append(parsed)
        except Exception as e:
            print(f"Error at sample {i}: {e}")
            y_pred.append(1)  # neutral fallback
            all_generations.append("ERROR")

    y_true = np.array(labels, dtype=np.int64)
    y_pred = np.array(y_pred, dtype=np.int64)

    # Debug: first 5 + every 100th
    for i in range(min(5, n)):
        print(f"Debug Sample {i}: Generated: {all_generations[i]}, Parsed Label: {y_pred[i]}")
    for i in range(100, n, 100):
        if i < n:
            print(f"Debug Sample {i}: Generated: {all_generations[i]}, Parsed Label: {y_pred[i]}")

    print("True label dist:", dict(Counter(y_true)))
    print("Pred label dist:", dict(Counter(y_pred)))

    return y_true, y_pred

In [6]:
def compute_metrics(y_true, y_pred):
    acc = float(accuracy_score(y_true, y_pred))
    f1_macro = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    f1_per_class = f1_score(y_true, y_pred, average=None, zero_division=0)
    f1_per_class = {LABEL_NAMES[i]: float(f1_per_class[i]) for i in range(NUM_LABELS)}
    cm = confusion_matrix(y_true, y_pred)
    out = {"accuracy": acc, "f1_macro": f1_macro, "f1_per_class": f1_per_class}
    return out, cm


def save_confusion_plot(cm, path):
    if not HAS_PLOT:
        return
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    plt.tight_layout()
    plt.savefig(path)
    plt.close()

In [7]:
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
all_metrics = {}

for config_name in CONFIGS:
    ds_dict = datasets[config_name]
    split_names = EVAL_SPLITS[config_name]
    all_metrics[config_name] = {}

    for split_name in split_names:
        if split_name not in ds_dict:
            print(f"  Skip {config_name}/{split_name} (missing)")
            continue
        ds = ds_dict[split_name]
        print(f"Evaluating {config_name} / {split_name} ...")
        y_true, y_pred = run_prompted_inference(ds)
        metrics, cm = compute_metrics(y_true, y_pred)
        all_metrics[config_name][split_name] = metrics

        cm_path = Path(RESULTS_DIR) / f"confusion_{config_name}_{split_name}.csv"
        np.savetxt(cm_path, cm, fmt="%d", delimiter=",")
        save_confusion_plot(cm, Path(RESULTS_DIR) / f"confusion_{config_name}_{split_name}.png")

        print(f"  accuracy={metrics['accuracy']:.4f}, f1_macro={metrics['f1_macro']:.4f}")

with open(Path(RESULTS_DIR) / "metrics.json", "w") as f:
    json.dump(all_metrics, f, indent=2)
print(f"Saved {RESULTS_DIR}/metrics.json")

Evaluating snli_tr_1_1 / test ...


Inference: 100%|██████████| 9824/9824 [16:56<00:00,  9.66it/s]


Debug Sample 0: Generated: , Parsed Label: 1
Debug Sample 1: Generated: , Parsed Label: 1
Debug Sample 2: Generated: , Parsed Label: 1
Debug Sample 3: Generated: , Parsed Label: 1
Debug Sample 4: Generated: , Parsed Label: 1
Debug Sample 100: Generated: , Parsed Label: 1
Debug Sample 200: Generated: , Parsed Label: 1
Debug Sample 300: Generated: , Parsed Label: 1
Debug Sample 400: Generated: , Parsed Label: 1
Debug Sample 500: Generated: , Parsed Label: 1
Debug Sample 600: Generated: , Parsed Label: 1
Debug Sample 700: Generated: , Parsed Label: 1
Debug Sample 800: Generated: , Parsed Label: 1
Debug Sample 900: Generated: , Parsed Label: 1
Debug Sample 1000: Generated: , Parsed Label: 1
Debug Sample 1100: Generated: , Parsed Label: 1
Debug Sample 1200: Generated: , Parsed Label: 1
Debug Sample 1300: Generated: , Parsed Label: 1
Debug Sample 1400: Generated: , Parsed Label: 1
Debug Sample 1500: Generated: , Parsed Label: 1
Debug Sample 1600: Generated: , Parsed Label: 1
Debug Sample 170

Inference: 100%|██████████| 9809/9809 [19:42<00:00,  8.30it/s]


Debug Sample 0: Generated: , Parsed Label: 1
Debug Sample 1: Generated: , Parsed Label: 1
Debug Sample 2: Generated: , Parsed Label: 1
Debug Sample 3: Generated: , Parsed Label: 1
Debug Sample 4: Generated: , Parsed Label: 1
Debug Sample 100: Generated: , Parsed Label: 1
Debug Sample 200: Generated: , Parsed Label: 1
Debug Sample 300: Generated: , Parsed Label: 1
Debug Sample 400: Generated: , Parsed Label: 1
Debug Sample 500: Generated: , Parsed Label: 1
Debug Sample 600: Generated: , Parsed Label: 1
Debug Sample 700: Generated: , Parsed Label: 1
Debug Sample 800: Generated: , Parsed Label: 1
Debug Sample 900: Generated: , Parsed Label: 1
Debug Sample 1000: Generated: , Parsed Label: 1
Debug Sample 1100: Generated: , Parsed Label: 1
Debug Sample 1200: Generated: , Parsed Label: 1
Debug Sample 1300: Generated: , Parsed Label: 1
Debug Sample 1400: Generated: , Parsed Label: 1
Debug Sample 1500: Generated: , Parsed Label: 1
Debug Sample 1600: Generated: , Parsed Label: 1
Debug Sample 170

...

Debug Sample 0: Generated: , Parsed Label: 1
Debug Sample 1: Generated: , Parsed Label: 1
Debug Sample 2: Generated: , Parsed Label: 1
Debug Sample 3: Generated: , Parsed Label: 1
Debug Sample 4: Generated: , Parsed Label: 1
Debug Sample 100: Generated: , Parsed Label: 1
Debug Sample 200: Generated: , Parsed Label: 1
Debug Sample 300: Generated: , Parsed Label: 1
Debug Sample 400: Generated: , Parsed Label: 1
Debug Sample 500: Generated: , Parsed Label: 1
Debug Sample 600: Generated: , Parsed Label: 1
Debug Sample 700: Generated: , Parsed Label: 1
Debug Sample 800: Generated: , Parsed Label: 1
Debug Sample 900: Generated: , Parsed Label: 1
Debug Sample 1000: Generated: , Parsed Label: 1
Debug Sample 1100: Generated: , Parsed Label: 1
Debug Sample 1200: Generated: , Parsed Label: 1
Debug Sample 1300: Generated: , Parsed Label: 1
Debug Sample 1400: Generated: , Parsed Label: 1
Debug Sample 1500: Generated: , Parsed Label: 1
Debug Sample 1600: Generated: , Parsed Label: 1
Debug Sample 170

Inference: 100%|██████████| 9008/9008 [18:07<00:00,  8.29it/s]


Debug Sample 0: Generated: , Parsed Label: 1
Debug Sample 1: Generated: , Parsed Label: 1
Debug Sample 2: Generated: , Parsed Label: 1
Debug Sample 3: Generated: , Parsed Label: 1
Debug Sample 4: Generated: , Parsed Label: 1
Debug Sample 100: Generated: , Parsed Label: 1
Debug Sample 200: Generated: , Parsed Label: 1
Debug Sample 300: Generated: , Parsed Label: 1
Debug Sample 400: Generated: , Parsed Label: 1
Debug Sample 500: Generated: , Parsed Label: 1
Debug Sample 600: Generated: , Parsed Label: 1
Debug Sample 700: Generated: , Parsed Label: 1
Debug Sample 800: Generated: , Parsed Label: 1
Debug Sample 900: Generated: , Parsed Label: 1
Debug Sample 1000: Generated: , Parsed Label: 1
Debug Sample 1100: Generated: , Parsed Label: 1
Debug Sample 1200: Generated: , Parsed Label: 1
Debug Sample 1300: Generated: , Parsed Label: 1
Debug Sample 1400: Generated: , Parsed Label: 1
Debug Sample 1500: Generated: , Parsed Label: 1
Debug Sample 1600: Generated: , Parsed Label: 1
Debug Sample 170

Inference: 100%|██████████| 9217/9217 [18:58<00:00,  8.10it/s]

Debug Sample 0: Generated: , Parsed Label: 1
Debug Sample 1: Generated: , Parsed Label: 1
Debug Sample 2: Generated: , Parsed Label: 1
Debug Sample 3: Generated: , Parsed Label: 1
Debug Sample 4: Generated: , Parsed Label: 1
Debug Sample 100: Generated: , Parsed Label: 1
Debug Sample 200: Generated: , Parsed Label: 1
Debug Sample 300: Generated: , Parsed Label: 1
Debug Sample 400: Generated: , Parsed Label: 1
Debug Sample 500: Generated: , Parsed Label: 1
Debug Sample 600: Generated: , Parsed Label: 1
Debug Sample 700: Generated: , Parsed Label: 1
Debug Sample 800: Generated: , Parsed Label: 1
Debug Sample 900: Generated: , Parsed Label: 1
Debug Sample 1000: Generated: , Parsed Label: 1
Debug Sample 1100: Generated: , Parsed Label: 1
Debug Sample 1200: Generated: , Parsed Label: 1
Debug Sample 1300: Generated: , Parsed Label: 1
Debug Sample 1400: Generated: , Parsed Label: 1
Debug Sample 1500: Generated: , Parsed Label: 1
Debug Sample 1600: Generated: , Parsed Label: 1
Debug Sample 170

In [8]:
# Summary: per config/split
for config_name, splits in all_metrics.items():
    for split_name, m in splits.items():
        print(f"{config_name} / {split_name}: acc={m['accuracy']:.4f}, F1_macro={m['f1_macro']:.4f}, F1_per_class={m['f1_per_class']}")

snli_tr_1_1 / test: acc=0.3277, F1_macro=0.1645, F1_per_class={'entailment': 0.0, 'neutral': 0.49359809859694853, 'contradiction': 0.0}
multinli_tr_1_1 / validation_matched: acc=0.3184, F1_macro=0.1610, F1_per_class={'entailment': 0.0, 'neutral': 0.4829879369007114, 'contradiction': 0.0}
multinli_tr_1_1 / validation_mismatched: acc=0.3185, F1_macro=0.1610, F1_per_class={'entailment': 0.0, 'neutral': 0.48309402501157944, 'contradiction': 0.0}
trglue_mnli / test_matched: acc=0.3484, F1_macro=0.1722, F1_per_class={'entailment': 0.0, 'neutral': 0.5167133212580274, 'contradiction': 0.0}
trglue_mnli / test_mismatched: acc=0.3302, F1_macro=0.1655, F1_per_class={'entailment': 0.0, 'neutral': 0.4964110929853181, 'contradiction': 0.0}
